In [1]:
import pandas as pd
df = pd.read_csv('VN_housing_dataset.csv')

In [2]:
df.head(10)

,Unnamed: 0,Ngày,Địa chỉ,Quận,Huyện,Loại hình nhà ở,Giấy tờ pháp lý,Số tầng,Số phòng ngủ,Diện tích,Dài,Rộng,Giá/m2
0,0.0,2020-08-05,"Đường Hoàng Quốc Việt, Phường Nghĩa Đô, Quận C...",Quận Cầu Giấy,Phường Nghĩa Đô,"Nhà ngõ, hẻm",Đã có sổ,4,5 phòng,46 m²,NaN,NaN,"86,96 triệu/m²"
1,1.0,2020-08-05,"Đường Kim Giang, Phường Kim Giang, Quận Thanh ...",Quận Thanh Xuân,Phường Kim Giang,"Nhà mặt phố, mặt tiền",NaN,NaN,3 phòng,37 m²,NaN,NaN,"116,22 triệu/m²"
2,2.0,2020-08-05,"phố minh khai, Phường Minh Khai, Quận Hai Bà T...",Quận Hai Bà Trưng,Phường Minh Khai,"Nhà ngõ, hẻm",Đã có sổ,4,4 phòng,40 m²,10 m,4 m,65 triệu/m²
3,3.0,2020-08-05,"Đường Võng Thị, Phường Thụy Khuê, Quận Tây Hồ,...",Quận Tây Hồ,Phường Thụy Khuê,"Nhà ngõ, hẻm",Đã có sổ,NaN,6 phòng,51 m²,12.75 m,4 m,100 triệu/m²
4,4.0,2020-08-05,"Đường Kim Giang, Phường Kim Giang, Quận Thanh ...",Quận Thanh Xuân,Phường Kim Giang,"Nhà ngõ, hẻm",NaN,NaN,4 phòng,36 m²,9 m,4 m,"86,11 triệu/m²"
5,5.0,2020-08-05,"Đường Yên Hòa, Phường Yên Hoà, Quận Cầu Giấy, ...",Quận Cầu Giấy,Phường Yên Hoà,"Nhà ngõ, hẻm",Đã có sổ,NaN,nhiều hơn 10 phòng,46 m²,12.1 m,3.8 m,"104,35 triệu/m²"
6,6.0,2020-08-05,"Đường Tây Sơn, Phường Trung Liệt, Quận Đống Đa...",Quận Đống Đa,Phường Trung Liệt,"Nhà ngõ, hẻm",NaN,NaN,3 phòng,52 m²,NaN,4.5 m,"112,5 triệu/m²"
7,7.0,2020-08-05,"Đường Lò Đúc, Phường Đống Mác, Quận Hai Bà Trư...",Quận Hai Bà Trưng,Phường Đống Mác,"Nhà mặt phố, mặt tiền",Đã có sổ,6,5 phòng,32 m²,NaN,6.8 m,"184,38 triệu/m²"
8,8.0,2020-08-05,"Đường Xuân La, Phường Xuân La, Quận Tây Hồ, Hà...",Quận Tây Hồ,Phường Xuân La,"Nhà ngõ, hẻm",NaN,NaN,4 phòng,75 m²,12 m,6.5 m,120 triệu/m²
9,9.0,2020-08-05,"Đường 19/5, Phường Văn Quán, Quận Hà Đông, Hà Nội",Quận Hà Đông,Phường Văn Quán,"Nhà ngõ, hẻm",Đã có sổ,4,3 phòng,41 m²,NaN,3.5 m,"64,63 triệu/m²"


In [3]:
df.shape

(82497, 13)

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error # Import thêm metric

# 1. Đọc dữ liệu
# Lưu ý: Đảm bảo file csv nằm cùng thư mục với file code
df = pd.read_csv('VN_housing_dataset.csv')

# ==============================================================================
# BƯỚC 1: SƠ CHẾ CƠ BẢN & LOẠI BỎ CỘT NHIỄU
# ==============================================================================

# Đổi tên cột
df_renamed = df.rename(columns={
    "Ngày": "date", "Địa chỉ": "address", "Quận": "district", 
    "Huyện": "ward", "Loại hình nhà ở": "type_of_housing",
    "Giấy tờ pháp lý": "legal_paper", "Số tầng": "num_floors",
    "Số phòng ngủ": "num_bed_rooms", "Diện tích": "squared_meter_area",
    "Giá/m2": "price_in_million_per_square_meter"
})

# XÓA CỘT KHÔNG CẦN THIẾT
cols_to_drop = ['Unnamed: 0', 'Dài', 'Rộng', 'length_meter', 'width_meter']
df_renamed = df_renamed.drop(columns=[c for c in cols_to_drop if c in df_renamed.columns], errors='ignore')

# XÓA DỮ LIỆU TRÙNG LẶP
print(f"Số dòng trước khi xóa trùng: {df_renamed.shape[0]}")
df_renamed = df_renamed.drop_duplicates()
print(f"Số dòng sau khi xóa trùng: {df_renamed.shape[0]}")

# ==============================================================================
# BƯỚC 2: LÀM SẠCH ĐỊNH DẠNG (FORMAT CLEANING)
# ==============================================================================

def clean_numeric_str(series, remove_suffix):
    return (series.astype(str)
            .str.replace(remove_suffix, '', regex=False)
            .str.replace(',', '.', regex=False)
            .str.strip()
            .replace({'nan': np.nan, 'NaN': np.nan, 'None': np.nan}))

# Làm sạch District/Ward
df_renamed['district'] = df_renamed['district'].str.replace('Quận', '').str.strip()
df_renamed['ward'] = df_renamed['ward'].str.replace('Phường', '').str.strip()

# --- BỔ SUNG: TÁCH TÊN ĐƯỜNG (STREET) ĐỂ DÙNG CHO BƯỚC ENCODING ---
# Giả định địa chỉ có dạng: "Số nhà, Tên đường, Phường..." -> Lấy phần tên đường
# (Logic đơn giản: Lấy phần text đầu tiên trước dấu phẩy)
df_renamed['street'] = df_renamed['address'].astype(str).str.split(',').str[0].str.strip()

# Làm sạch Số tầng & Phòng ngủ
df_renamed['num_floors'] = df_renamed['num_floors'].replace('Nhiều hơn 10', '11')
df_renamed['num_bed_rooms'] = df_renamed['num_bed_rooms'].replace('nhiều hơn 10 phòng', '11')

df_renamed['num_floors'] = pd.to_numeric(clean_numeric_str(df_renamed['num_floors'], ' tầng'), errors='coerce')
df_renamed['num_bed_rooms'] = pd.to_numeric(clean_numeric_str(df_renamed['num_bed_rooms'], ' phòng'), errors='coerce')
df_renamed['squared_meter_area'] = pd.to_numeric(clean_numeric_str(df_renamed['squared_meter_area'], ' m²'), errors='coerce')

# Xử lý Cột GIÁ
def parse_price(val):
    if pd.isna(val): return np.nan
    val = str(val).lower()
    
    multiplier = 1
    if 'tỷ/m²' in val:
        multiplier = 1000
    elif 'đ/m²' in val:
        multiplier = 0.000001
    
    clean_val = val.replace('tỷ/m²', '').replace('triệu/m²', '').replace('đ/m²', '').replace('.', '').replace(',', '.')
    
    try:
        return float(clean_val) * multiplier
    except:
        return np.nan

df_renamed['price_in_million_per_square_meter'] = df_renamed['price_in_million_per_square_meter'].apply(parse_price)

# ==============================================================================
# BƯỚC 3: XỬ LÝ NGOẠI LAI & LOGIC DỮ LIỆU
# ==============================================================================

# 1. Lọc cứng
mask_valid = (
    (df_renamed['squared_meter_area'] >= 10) & (df_renamed['squared_meter_area'] <= 1000) & 
    (df_renamed['price_in_million_per_square_meter'] >= 5) & (df_renamed['price_in_million_per_square_meter'] <= 1000) & 
    (df_renamed['num_floors'] <= 15)
)

df_clean = df_renamed[mask_valid].copy()

# 2. Lọc IQR theo nhóm
def remove_outliers_iqr(group):
    if len(group) < 5: return group
    Q1 = group['price_in_million_per_square_meter'].quantile(0.25)
    Q3 = group['price_in_million_per_square_meter'].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 3.0 * IQR 
    upper = Q3 + 3.0 * IQR
    return group[(group['price_in_million_per_square_meter'] >= lower) & (group['price_in_million_per_square_meter'] <= upper)]

# Lưu ý: include_groups=False là mặc định trong pandas mới, hoặc xử lý để tránh warning
df_outlier_handled = df_clean.groupby(['district', 'type_of_housing']).apply(remove_outliers_iqr).reset_index(drop=True)

# ==============================================================================
# BƯỚC 4: ĐIỀN DỮ LIỆU THIẾU
# ==============================================================================

cols_to_fill = ['num_floors', 'num_bed_rooms']

for col in cols_to_fill:
    df_outlier_handled[col] = df_outlier_handled.groupby(['district', 'type_of_housing'])[col].transform(lambda x: x.fillna(x.median()))
    df_outlier_handled[col] = df_outlier_handled.groupby('district')[col].transform(lambda x: x.fillna(x.median()))
    df_outlier_handled[col] = df_outlier_handled[col].fillna(df_outlier_handled[col].median())

df_outlier_handled['num_floors'] = df_outlier_handled['num_floors'].round().astype(int)
df_outlier_handled['num_bed_rooms'] = df_outlier_handled['num_bed_rooms'].round().astype(int)
df_outlier_handled['legal_paper'] = df_outlier_handled['legal_paper'].fillna("Chưa rõ")

final_df = df_outlier_handled.reset_index(drop=True)
print("\nThống kê dữ liệu cuối cùng:")
print(final_df.info())
print("-" * 30)
print(final_df.describe())

final_df.to_csv('housing_data_cleaned_v3.csv', index=False, encoding='utf-8-sig')
print("\nĐã lưu file: housing_data_cleaned_v3.csv")

Số dòng trước khi xóa trùng: 82497
Số dòng sau khi xóa trùng: 81439


C:\Users\Cong Thanh\AppData\Local\Temp\ipykernel_17148\72353597.py:106: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_outlier_handled = df_clean.groupby(['district', 'type_of_housing']).apply(remove_outliers_iqr).reset_index(drop=True)



Thống kê dữ liệu cuối cùng:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35257 entries, 0 to 35256
Data columns (total 11 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   date                               35257 non-null  object 
 1   address                            35256 non-null  object 
 2   district                           35257 non-null  object 
 3   ward                               35251 non-null  object 
 4   type_of_housing                    35257 non-null  object 
 5   legal_paper                        35257 non-null  object 
 6   num_floors                         35257 non-null  int64  
 7   num_bed_rooms                      35257 non-null  int64  
 8   squared_meter_area                 35257 non-null  float64
 9   price_in_million_per_square_meter  35257 non-null  float64
 10  street                             35257 non-null  object 
dtypes: float64(2), int64(2), 